# Required 31B official-harness workbench
        Run preflight, validate the package, then invoke the organizer's documented command.
        No unverified loader API, guessed flags, E2B fallback or fake score is supplied.
        Attach the official competition data and read HARNESS_README.md before configuring the runner.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, sys, zipfile
from datetime import datetime, timezone

# Set a full archive path if multiple matching datasets are attached.
WORKBENCH_ZIP = None
if WORKBENCH_ZIP is None:
    candidates = list(Path('/kaggle/input').rglob('kaggle_phase2_workbench.zip')) if Path('/kaggle/input').exists() else []
    if not candidates:
        candidates = list(Path.cwd().glob('output/kaggle_phase2_workbench.zip'))
    if len(candidates) != 1:
        raise RuntimeError('Attach the workbench dataset and set WORKBENCH_ZIP to its exact path')
    WORKBENCH_ZIP = candidates[0]
WORKBENCH_ZIP = Path(WORKBENCH_ZIP)
execution_environment = 'kaggle' if Path('/kaggle/working').exists() else 'local_not_kaggle'
run_base = Path('/kaggle/working') if execution_environment == 'kaggle' else Path.cwd() / 'scratch/kaggle_notebook_checks'
RUN = run_base / ('neuralos_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
ROOT = RUN / 'workbench'
ROOT.mkdir(parents=True)
with zipfile.ZipFile(WORKBENCH_ZIP) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)): raise ValueError('Duplicate archive members')
    for item in archive.infolist():
        target = (ROOT / item.filename).resolve()
        if item.is_dir() or not target.is_relative_to(ROOT.resolve()): raise ValueError('Invalid archive path')
        if (item.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError('Symlink archive member')
    manifest = json.loads(archive.read('manifest.json'))
    if set(names) != {'manifest.json'} | {item['path'] for item in manifest['files']}: raise ValueError('Manifest/file mismatch')
    for item in manifest['files']:
        data = archive.read(item['path'])
        if hashlib.sha256(data).hexdigest() != item['sha256']: raise ValueError('Hash mismatch: ' + item['path'])
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
print('Verified archive:', hashlib.sha256(WORKBENCH_ZIP.read_bytes()).hexdigest())
print('Execution environment:', execution_environment)
print('Output directory:', RUN)


In [ ]:
import importlib.metadata, subprocess, time
environment = {'execution_environment': execution_environment, 'platform': platform.platform(),
               'python': sys.version, 'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'packages': {}}
for package in ('torch', 'transformers', 'vllm', 'PyYAML', 'jsonschema'):
    try: environment['packages'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: environment['packages'][package] = None
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader,nounits'],
                         capture_output=True, text=True, timeout=15)
    environment['gpu_query'] = {'returncode': gpu.returncode, 'stdout': gpu.stdout, 'stderr': gpu.stderr}
except (OSError, subprocess.TimeoutExpired) as error: environment['gpu_query'] = {'unavailable': str(error)}
(RUN / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))


In [ ]:
from scripts.build_competition_submission import validate_submission, build_submission
agent, reachable = validate_submission(ROOT / 'competition')
submission = RUN / 'submission.zip'
package_manifest = build_submission(ROOT / 'competition', submission)
agent_submission_dir = RUN / 'agent_submission'
agent_submission_dir.mkdir()
with zipfile.ZipFile(submission) as archive: archive.extractall(agent_submission_dir)
print(json.dumps(package_manifest, indent=2))
print('Validation level: conservative checks against official README; organizer compiler not yet run')


In [ ]:
official_readmes = list(Path('/kaggle/input').rglob('HARNESS_README.md')) if Path('/kaggle/input').exists() else []
provenance = [{'path': str(p), 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in official_readmes]
(RUN / 'official_harness_provenance.json').write_text(json.dumps(provenance, indent=2))
print('Official README candidates:', json.dumps(provenance, indent=2))
if len(official_readmes) == 1: print(official_readmes[0].read_text())
else: print('Select the actual official files; DeepSeek reference files are not substitutes.')


## Configure the actual organizer command
        Read the preceding official instructions and install their pinned dependencies if necessary.
        Set the argument list below exactly as documented, with actual model/dataset/submission paths.
        The command below follows the downloaded official README. Install only verified organizer
        packages in a preparation session and attach them for the offline L4 session. The notebook captures raw
        output; it does not infer resolved issues from a process return code.


In [ ]:
config = json.loads((ROOT / 'kaggle/phase2_run_config.json').read_text())
import shutil
from competition.study import prepare_study
RUN_OFFICIAL_EVAL = False  # Explicitly enable in the configured Kaggle session.
OFFICIAL_RUN_ARGV = OFFICIAL_RUN_CWD = OFFICIAL_RESULT_PATH = None
if len(official_readmes) == 1:
    data_dir = official_readmes[0].parent
    tasks_path, snapshots_path = data_dir / 'tasks.jsonl', data_dir / 'snapshots'
    if not tasks_path.is_file() or not snapshots_path.is_dir():
        raise RuntimeError('Official task manifest and snapshots must be attached together')
    study = prepare_study(tasks_path, RUN / 'locked_study', seed=20261003,
        protocol=ROOT / 'paper/SWE_MEMORY_STUDY_PROTOCOL.md',
        method_files=[ROOT / 'competition/agent.yaml', ROOT / 'competition/prompts/system.md',
                      ROOT / 'competition/configs/sampling.yaml', ROOT / 'memory/joint_resource_calibration.py'],
        preexposed_reference_ids={'fastapi_15661'})
    # One development issue, chosen without its answer, for initial harness smoke validation.
    smoke_id = study['folds'][0]['tasks']['fit'][0]
    OFFICIAL_RUN_ARGV = ['swegemma', 'eval', '--tasks', str(tasks_path),
        '--snapshots-dir', str(snapshots_path), '--submission-dir', str(agent_submission_dir),
        '--results-dir', str(RUN / 'official_results'), '--sandbox', 'subprocess',
        '--max-tool-calls', '50', '--max-time-minutes', '30', '--concurrency', '1',
        '--display', 'auto', '--task-id', smoke_id]
    OFFICIAL_RUN_CWD = str(RUN)
    OFFICIAL_RESULT_PATH = str(RUN / 'official_results/summary.json')
config.update(run_official_eval=RUN_OFFICIAL_EVAL, runner_available=shutil.which('swegemma') is not None)
config.update(official_run_argv=OFFICIAL_RUN_ARGV, official_run_cwd=OFFICIAL_RUN_CWD,
              official_result_path=OFFICIAL_RESULT_PATH)
if agent['model'] != config['required_model']: raise ValueError('Required 31B base model mismatch')
(RUN / 'run_config.json').write_text(json.dumps(config, indent=2))


In [ ]:
outcome = {'status': 'configured_not_executed' if OFFICIAL_RUN_ARGV else 'official_data_not_attached', 'score': None, 'issue_results': None,
           'submission_sha256': package_manifest['archive_sha256'], 'required_model': config['required_model']}
if RUN_OFFICIAL_EVAL and config['runner_available'] and OFFICIAL_RUN_ARGV and OFFICIAL_RUN_CWD:
    if not isinstance(OFFICIAL_RUN_ARGV, list) or not all(isinstance(x, str) for x in OFFICIAL_RUN_ARGV):
        raise ValueError('Runner must be an explicit argument list')
    started = time.monotonic()
    try:
        with (RUN / 'official_stdout.log').open('w') as stdout, (RUN / 'official_stderr.log').open('w') as stderr:
            process = subprocess.run(OFFICIAL_RUN_ARGV, cwd=OFFICIAL_RUN_CWD, stdout=stdout, stderr=stderr,
                                     timeout=config['max_process_seconds'], check=False)
        outcome.update(status='process_completed' if process.returncode == 0 else 'process_failed', returncode=process.returncode)
    except subprocess.TimeoutExpired:
        outcome.update(status='process_timeout', returncode=None)
    except OSError as error:
        outcome.update(status='runner_start_failed', error=str(error), returncode=None)
    outcome['wall_seconds'] = time.monotonic() - started
    if OFFICIAL_RESULT_PATH and Path(OFFICIAL_RESULT_PATH).is_file():
        result = Path(OFFICIAL_RESULT_PATH)
        outcome['official_result'] = {'path': str(result), 'sha256': hashlib.sha256(result.read_bytes()).hexdigest()}
(RUN / 'official_run_outcome.json').write_text(json.dumps(outcome, indent=2))
print(json.dumps(outcome, indent=2))


## Next experiment
        Verify actual patch validation, runtime budget and model/adapter loading before claiming a score.
        The frozen study is built from real attached tasks, with references excluded from agent views.
        Its evaluator directory must stay outside sandbox mounts. Compare no-adapter baseline,
        retrieval/checkpoints and adapter changes separately.
        Failed and unattempted issues stay in the denominator. Process logs alone are not a leaderboard
        estimate. No adapter training has been performed by this notebook.
